# DiVaST (NLBSE'27): official evaluation of the distillation models on a Colab T4

Inference only. Evaluates every model in `My Drive/divast_t4/models/` (the teacher
and seven DiVaST students) and the organisers' baseline with the official protocol
(batch 600, 10 timed repetitions, threshold 0.5, `torch.profiler` FLOPs). All
models run in this one session, so their runtimes are comparable.

**Run**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Runtime > **Run all**, and allow Google Drive access.

**Disconnects.** Every finished model is saved to `My Drive/divast_t4/results/`
immediately. If the session drops, reconnect and Run all again: finished models
are skipped, so only the model in progress is repeated.

**Time.** About 2 h in total: the submitted student, the baseline (about 40 min)
and the teacher (about 20 min) run first, then the other students (3 to 15 min
each). Keep this tab open and check it about every 30 min, because Colab
disconnects sessions it considers idle.

In [ ]:
# 1. Check the GPU and install packages.
import subprocess, importlib.metadata as md

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
assert "T4" in gpu, "Not a T4. Runtime > Change runtime type > T4 GPU, then Run all again."

# Same as the organisers' notebook: Colab's own torch and transformers, plus datasets.
# The models were saved with transformers 5.x, so upgrade only if Colab ships 4.x.
try:
    tv = md.version("transformers")
except md.PackageNotFoundError:
    tv = "0"
pkgs = ["datasets"]
if int(tv.split(".")[0]) < 5:
    pkgs.append("transformers==5.22.1")
subprocess.run(["pip", "-q", "install", *pkgs], check=True)
print("torch", md.version("torch"), "| transformers", md.version("transformers"),
      "| datasets", md.version("datasets"))

In [ ]:
# 2. Mount Google Drive. Inputs and results live there.
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/divast_t4'
MODELS = os.path.join(WORK, 'models')
RESULTS = os.path.join(WORK, 'results')
assert os.path.isdir(MODELS), f"{MODELS} not found. Upload the divast_t4 folder to the top level of My Drive."
os.makedirs(RESULTS, exist_ok=True)

SRC_DIR = '/content/src'                       # evaluation code (rewritten each session)
LOCAL = '/content/models_local'                # models are copied here before loading
os.makedirs(SRC_DIR, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'    # dataset and baseline cache on local disk

# Official protocol. Do not change.
BATCH, REPS, LIMIT = 600, 10, 0

## 3. Evaluation code
The same tested scripts as the local pipeline.

In [ ]:
%%writefile /content/src/common.py
"""Shared constants and helpers for the NLBSE'27 multitask code classification pipeline.

The task: one function string -> two binary labels in order [Vulnerability, MAT/SATD].
Reference: the organisers' repository https://github.com/nlbse2027/code-comment-classification
(README.md: data description; CodeBERT_baseline.ipynb: evaluation loop and score).
"""
from __future__ import annotations

import os
import random
from dataclasses import dataclass

import numpy as np

# ---------------------------------------------------------------------------
# Dataset / label constants
# ---------------------------------------------------------------------------
HF_DATASET = "NLBSE/nlbse27-code-comment-classification"
BASELINE_MODEL = "NLBSE/nlbse27-code-comment-classification-baseline"

# Label order is fixed by the organisers' evaluation notebook:
#   index 0 -> Vulnerability / weakness, index 1 -> MAT (SATD proxy).
LABEL_NAMES = ["Vulnerability", "MAT"]
TEXT_COLUMN = "Function"
LABEL_COLUMN = "Label"
NUM_LABELS = 2

# ---------------------------------------------------------------------------
# Official scoring formula (verbatim from CodeBERT_baseline.ipynb, cell 4)
# ---------------------------------------------------------------------------
MAX_AVG_RUNTIME = 1000       # seconds, one full pass over the test split
MAX_AVG_FLOPS = 3_000_000    # GFLOPs, one full pass over the test split


def submission_score(avg_f1: float, avg_runtime: float, avg_flops: float) -> float:
    """Composite competition score. Higher is better; max is 1.0.

    score = 0.6 * mean_F1
          + 0.2 * max((MAX_AVG_RUNTIME - avg_runtime) / MAX_AVG_RUNTIME, 0)
          + 0.2 * max((MAX_AVG_FLOPS  - avg_flops)   / MAX_AVG_FLOPS,   0)
    """
    return (
        0.6 * avg_f1
        + 0.2 * max((MAX_AVG_RUNTIME - avg_runtime) / MAX_AVG_RUNTIME, 0.0)
        + 0.2 * max((MAX_AVG_FLOPS - avg_flops) / MAX_AVG_FLOPS, 0.0)
    )


# ---------------------------------------------------------------------------
# Misc helpers
# ---------------------------------------------------------------------------
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def per_task_prf1(predictions: np.ndarray, labels: np.ndarray):
    """Per-task precision/recall/F1 computed exactly like the organisers' loop."""
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    out = []
    for idx, name in enumerate(LABEL_NAMES):
        p = predictions[:, idx]
        y = labels[:, idx]
        tp = int(np.sum((p == 1) & (y == 1)))
        fp = int(np.sum((p == 1) & (y == 0)))
        fn = int(np.sum((p == 0) & (y == 1)))
        tn = int(np.sum((p == 0) & (y == 0)))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        out.append({
            "cat": name, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": precision, "recall": recall, "f1": f1,
        })
    return out


@dataclass
class Paths:
    root: str = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

    @property
    def data(self):
        return os.path.join(self.root, "data")

    @property
    def models(self):
        return os.path.join(self.root, "models")

    @property
    def results(self):
        return os.path.join(self.root, "results")


PATHS = Paths()

# Keep the HF cache inside the project so everything is self-contained / documented.
os.environ.setdefault("HF_HOME", os.path.join(PATHS.root, ".hf_cache"))


In [ ]:
%%writefile /content/src/evaluate_submission.py
"""Evaluate a model with the EXACT organisers' protocol and compute the
competition score. Faithful re-implementation of CodeBERT_baseline.ipynb in
https://github.com/nlbse2027/code-comment-classification
(cells 3-4): same batching, repetitions, fixed 0.5 threshold, torch.profiler
FLOP counting, and scoring formula.

Usage:
  # Official protocol on the public test split:
  python src/evaluate_submission.py --model models/codeberta_small_cal
  # The organisers' published baseline, for reference:
  python src/evaluate_submission.py --model NLBSE/nlbse27-code-comment-classification-baseline
  # Fast smoke test (CPU, tiny):
  python src/evaluate_submission.py --model models/smoke --limit 64 \
      --batch-size 16 --repetitions 2 --no-amp
"""
from __future__ import annotations

import argparse
import contextlib
import json
import os
import time

import numpy as np

from common import (HF_DATASET, LABEL_COLUMN, PATHS, TEXT_COLUMN,
                    per_task_prf1, submission_score)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="HF id or local dir")
    ap.add_argument("--split", default="test")
    ap.add_argument("--batch-size", type=int, default=600)
    ap.add_argument("--repetitions", type=int, default=10)
    ap.add_argument("--max-length", type=int, default=512)
    ap.add_argument("--limit", type=int, default=0, help="cap #instances (smoke)")
    ap.add_argument("--no-amp", action="store_true", help="disable fp16 autocast")
    ap.add_argument("--out", default="")
    args = ap.parse_args()

    import torch
    from datasets import load_dataset
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    tokenizer = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForSequenceClassification.from_pretrained(args.model).to(device).eval()

    ds = load_dataset(HF_DATASET)
    test = ds[args.split].to_pandas()
    if args.limit > 0:
        test = test.iloc[:args.limit].reset_index(drop=True)
    print(f"Evaluating on {len(test)} instances of split '{args.split}'")

    tokenized = tokenizer(
        test[TEXT_COLUMN].tolist(), return_tensors="pt",
        padding=True, truncation=True, max_length=args.max_length,
    )
    dataset = TensorDataset(tokenized["input_ids"], tokenized["attention_mask"])
    loader = DataLoader(dataset, batch_size=args.batch_size)

    use_amp = (device.type == "cuda") and (not args.no_amp)
    predictions, total_time, total_flops = [], 0.0, 0.0

    for batch in loader:
        input_ids, attention_mask = (b.to(device, non_blocking=True) for b in batch)
        amp_ctx = torch.autocast(device_type="cuda") if use_amp else contextlib.nullcontext()
        begin = time.time()
        with torch.no_grad(), amp_ctx:
            with torch.profiler.profile(with_flops=True) as prof:
                for _ in range(args.repetitions):
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        if device.type == "cuda":
            torch.cuda.synchronize()
        total_time += time.time() - begin
        total_flops += sum(k.flops for k in prof.key_averages()) / 1e9

        probs = torch.sigmoid(outputs.logits)
        predictions.extend((probs > 0.5).int().cpu().numpy().tolist())
        del input_ids, attention_mask, outputs, probs
        if device.type == "cuda":
            torch.cuda.empty_cache()

    labels = np.array(test[LABEL_COLUMN].tolist())
    predictions = np.array(predictions)
    rows = per_task_prf1(predictions, labels)

    avg_runtime = total_time / args.repetitions
    avg_flops = total_flops / args.repetitions
    avg_f1 = float(np.mean([r["f1"] for r in rows]))
    avg_p = float(np.mean([r["precision"] for r in rows]))
    avg_r = float(np.mean([r["recall"] for r in rows]))
    score = submission_score(avg_f1, avg_runtime, avg_flops)

    print("\n---------------- RESULTS ----------------")
    for r in rows:
        print(f"{r['cat']:>14}:  P={r['precision']:.4f}  R={r['recall']:.4f}  F1={r['f1']:.4f}")
    print(f"{'mean':>14}:  P={avg_p:.4f}  R={avg_r:.4f}  F1={avg_f1:.4f}")
    print(f"Avg runtime (s):   {avg_runtime:.4f}")
    print(f"Avg compute GFLOPs:{avg_flops:.1f}")
    print(f"COMPETITION SCORE: {score:.4f}")

    result = {
        "model": args.model, "split": args.split, "n": int(len(test)),
        "batch_size": args.batch_size, "repetitions": args.repetitions,
        "max_length": args.max_length, "amp": use_amp,
        "per_task": rows, "mean_f1": avg_f1, "mean_precision": avg_p,
        "mean_recall": avg_r, "avg_runtime_s": avg_runtime,
        "avg_gflops": avg_flops, "score": score,
    }
    out = args.out or os.path.join(
        PATHS.results, "eval_" + os.path.basename(args.model.rstrip("/")) + ".json")
    os.makedirs(PATHS.results, exist_ok=True)
    with open(out, "w") as f:
        json.dump(result, f, indent=2)
    print("Wrote", out)


if __name__ == "__main__":
    main()


In [ ]:
# 4. Find the uploaded models and check that each upload is complete.
import json

BASELINE = "NLBSE/nlbse27-code-comment-classification-baseline"
PRIORITY = ["divast_3L_768H", "teacher", "divast_4L_768H", "divast_2L_768H", "divast_1L_768H", "divast_6L_512H", "divast_6L_384H", "divast_3L_384H"]
FIRST = ["divast_3L_768H", "divast_2L_768H", "baseline", "teacher"]

def upload_problem(d):
    """Compare the uploaded files with the sizes recorded before upload."""
    mf = os.path.join(d, '_manifest.json')
    if not os.path.exists(mf):
        return 'no _manifest.json (folder not made by collect_for_drive.py?)'
    with open(mf) as f:
        expected = json.load(f)
    for name, size in expected.items():
        p = os.path.join(d, name)
        if not os.path.exists(p):
            return f'missing {name}: upload not finished, upload this folder again'
        if os.path.getsize(p) != size:
            return f'{name} is {os.path.getsize(p)} bytes, expected {size}: upload this folder again'
    return None

def result_path(name):
    return os.path.join(RESULTS, f'eval_{name}.json')

queue = []
for name in sorted(os.listdir(MODELS)):
    d = os.path.join(MODELS, name)
    if not os.path.isdir(d):
        continue
    problem = upload_problem(d)
    if problem:
        print(f'SKIPPING {name}: {problem}')
        continue
    with open(os.path.join(d, 'config.json')) as f:
        layers = json.load(f).get('num_hidden_layers', 12)
    rank = PRIORITY.index(name) if name in PRIORITY else len(PRIORITY)
    queue.append((layers, rank, name, d))
queue.append((12, len(PRIORITY), 'baseline', BASELINE))
# FIRST in its order, then the other students.
queue.sort(key=lambda q: (0, FIRST.index(q[2])) if q[2] in FIRST else (1, q[0], q[1]))

print(f"{'model':<24}{'layers':>7}  status")
for layers, _, name, _src in queue:
    status = 'done (saved on Drive)' if os.path.exists(result_path(name)) else 'to do'
    print(f'{name:<24}{layers:>7}  {status}')

In [ ]:
# 5. Evaluate each model. Each result is saved to Drive as soon as it is done.
import csv, shutil, sys, time, datetime

ENV = {'gpu': gpu, 'torch': md.version('torch'),
       'transformers': md.version('transformers'), 'datasets': md.version('datasets')}

def write_leaderboard():
    rows = []
    for f in sorted(os.listdir(RESULTS)):
        if f.startswith('eval_') and f.endswith('.json'):
            with open(os.path.join(RESULTS, f)) as fh:
                r = json.load(fh)
            f1 = {t['cat']: t['f1'] for t in r['per_task']}
            rows.append({'model': r['model'], 'f1_vuln': round(f1['Vulnerability'], 4),
                         'f1_mat': round(f1['MAT'], 4), 'mean_f1': round(r['mean_f1'], 4),
                         'runtime_s': round(r['avg_runtime_s'], 2),
                         'gflops': round(r['avg_gflops']), 'score': round(r['score'], 4)})
    rows.sort(key=lambda r: -r['score'])
    if rows:
        tmp = os.path.join(RESULTS, 'leaderboard.csv.tmp')
        with open(tmp, 'w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=list(rows[0]))
            w.writeheader()
            w.writerows(rows)
        os.replace(tmp, os.path.join(RESULTS, 'leaderboard.csv'))
    return rows

local_results = os.path.join(os.path.dirname(SRC_DIR), 'results')
for layers, _, name, src in queue:
    if os.path.exists(result_path(name)):
        print(f'[skip] {name}: already saved on Drive')
        continue
    model_arg = src
    if os.path.isdir(src):                  # copy from Drive to local disk first
        model_arg = os.path.join(LOCAL, name)
        shutil.rmtree(model_arg, ignore_errors=True)
        shutil.copytree(src, model_arg)
    tmp_out = os.path.join(local_results, f'eval_{name}.json')
    cmd = [sys.executable, '-u', os.path.join(SRC_DIR, 'evaluate_submission.py'),
           '--model', model_arg, '--batch-size', str(BATCH),
           '--repetitions', str(REPS), '--out', tmp_out]
    if LIMIT:
        cmd += ['--limit', str(LIMIT)]
    t0 = time.time()
    print(f"\n=== {name} ({layers} layers), started {time.strftime('%H:%M:%S')} ===", flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        print(f'[FAILED] {name}: see the error above. Continuing with the next model.')
        continue
    with open(tmp_out) as f:
        res = json.load(f)
    res['model'] = name
    res['source'] = src
    res['env'] = {**ENV, 'finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
                  'wall_clock_min': round((time.time() - t0) / 60, 1)}
    tmp = result_path(name) + '.tmp'        # write then rename: never a half-written file
    with open(tmp, 'w') as f:
        json.dump(res, f, indent=2)
    os.replace(tmp, result_path(name))
    write_leaderboard()
    if os.path.isdir(src):
        shutil.rmtree(model_arg, ignore_errors=True)
    print(f"SAVED to Drive: {result_path(name)}  (score {res['score']:.4f})", flush=True)

print('\nAll models processed.')

In [ ]:
# 6. Leaderboard (also saved as My Drive/divast_t4/results/leaderboard.csv).
rows = write_leaderboard()
print(f"{'model':<24}{'F1 vuln':>9}{'F1 MAT':>9}{'mean F1':>9}{'runtime s':>11}{'GFLOPS':>11}{'score':>8}")
for r in rows:
    print(f"{r['model']:<24}{r['f1_vuln']:>9.4f}{r['f1_mat']:>9.4f}{r['mean_f1']:>9.4f}"
          f"{r['runtime_s']:>11.2f}{r['gflops']:>11,}{r['score']:>8.4f}")
missing = [n for _, _, n, _ in queue if not os.path.exists(result_path(n))]
print('\nStill missing:', ', '.join(missing) if missing else 'none. All done.')

In [ ]:
# 7. Optional, at the very end: force every pending write to reach Drive.
# Results are already synced in the background; this only makes certain of it.
drive.flush_and_unmount()
print('Drive flushed. Results are in My Drive/divast_t4/results/.')